# 自定义中间件-Wrap-style hooks
## 1、wrap_model_call的使用

### 1.1 基于装饰器的实现

In [1]:
from langchain.messages import HumanMessage
from langchain.agents.middleware import AgentMiddleware,wrap_model_call,ModelRequest,ModelResponse
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from typing import Callable
from langchain.agents import create_agent
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver



@wrap_model_call
def wrap_model_call_middleware(request:ModelRequest,handler:Callable[[ModelRequest],ModelResponse])-> ModelResponse | None:
    request.messages[-1].content += "---> wrap_model_call_before <---"
    # 模型的调用
    response = handler(request)
    response.result[0].content += "---> wrap_model_call_after <---"

    return response

# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    model="483025889/qwen3.5:9b", # ok
    # model="gemma4:12b", # 这里不好用
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 2.创建agent
agent = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware
    ]
)

resp = agent.invoke({"messages":[HumanMessage("你好")]})
for msg in resp['messages']:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！看起来你提到了 `wrap_model_call_before`，这可能是一个系统函数名称或工作流标识符。

不过我这边没有具体的上下文信息（比如是哪个平台/系统的具体模块）。为了更好地帮助你：

| 可能含义 | 说明 |
|---------|------|
| **函数标识** | 可能是某代码库中的包装器函数 |
| **流程节点** | 可能是编排/工作流系统的处理环节 |
| **API 端点** | 可能是后端服务的接口名之一 |

如果你能告诉我：
- 这是哪个系统/平台的功能？
- 你遇到了什么问题或有什么需求？

我会更具体地帮助你！😊---> wrap_model_call_after <---


### 1.2 基于类的实现

In [3]:
from langchain.messages import HumanMessage
from langchain.agents.middleware import AgentMiddleware,wrap_model_call,ModelRequest,ModelResponse
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from typing import Callable
from langchain.agents import create_agent
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver

class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self,request:ModelRequest,handler:Callable[[ModelRequest],ModelResponse])-> ModelResponse | None:
        request.messages[-1].content += "---> wrap_model_call_before <---"
        # 模型的调用
        response = handler(request)
        response.result[0].content += "---> wrap_model_call_after <---"

        return response
# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    model="483025889/qwen3.5:9b", # ok
    # model="gemma4:12b", # 这里不好用
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

wc_middleware = WrapModelCallMiddleware()

# 2.创建agent
agent = create_agent(
    model=model,
    middleware=[
        wc_middleware
    ]
)

resp = agent.invoke({"messages":[HumanMessage("你好")]})
for msg in resp['messages']:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！你提到的 `wrap_model_call_before` 和 `wrap_model_call_after` 看起来可能存在拼写或逻辑上的混淆。以下是可能的解决方案：

---

### 1. **确认函数名是否正确**
   - 如果项目中同时存在 `wrap_model_call_before` 和 `wrap_model_call_after`，请确保它们的调用顺序符合预期（比如 `before` 应在处理前执行，`after` 在处理后执行）。
   - 检查是否有拼写错误，例如：
     ```python
     # ❌ 错误的用法（重复的 before）
     wrap_model_call_before_wrap_model_call_after
   
     # ✅ 正确用法（按设计意图）
     def my_handler(event, context):
         ...
     
     events = [
       event('wrap_model_call_before').trigger(handler=my_handler),
       event('wrap_model_call_after').trigger(handler=my_handler)
     ]
     ```

---

### 2. **需求澄清**
   - **场景**：你是在开发事件驱动架构，还是构建装饰器？
   - **框架**：是否基于特定框架（如 AWS Lambda 事件源、FastAPI 等）？不同框架的命名规范可能不同。
   - **问题**：是否需要将两个逻辑合并为一个统一的钩子函数？例如 `wrap_model_call` + `before/after` 装饰器？

---

### 3. *